## Options Pricing with the Black–Scholes Model

This project downloads real **options market data** and computes theoretical option prices using the **Black–Scholes model**.

### Data Collection

The script retrieves market data from **Yahoo Finance** using the `yfinance` library.  
For a selected stock symbol, it downloads:

- All available **option expiration dates**
- **Call and put option chains**
- The current **stock price**
- The **risk-free interest rate** (from the 13-week Treasury bill, `^IRX`)

Calls and puts are merged into a single dataset containing:

- strike price
- implied volatility
- expiration date
- days to expiration
- option type (call or put)

### Black–Scholes Pricing

The project implements the **Black–Scholes option pricing formula** to compute the theoretical price of each option.

For each option, the model uses:

- **S** – current stock price  
- **K** – strike price  
- **T** – time to expiration (in years)  
- **r** – risk-free interest rate  
- **σ** – implied volatility  

From these parameters, the model calculates d<sub>1</sub> and d<sub>2</sub> and then derives the theoretical **call or put price**.

### Output

The final dataset includes:

- all downloaded **option market data**
- the computed **Black–Scholes theoretical price (`bsPrice`)**

### Purpose

This project demonstrates how to:

- Collect **real financial market data**
- Work with **options chains**
- Implement the **Black–Scholes pricing model**
- Compare **market prices with theoretical values**

It can serve as a foundation for further experiments in **quantitative finance**, such as volatility analysis, options mispricing detection, or trading strategy development.

In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.stats import norm

In [24]:
symbol = "AAPL"

In [38]:
def get_data(symbol):
  def process_expiration(exp_td_str):
    """
    Download Yahoo Finance call and put option quotes 
    for a single expiration
    Input:
    exp_td_str = expiration date string "%Y-$m-$d" 
        (a single item from yfinance.Ticker.options tuple)
    Return pandas.DataFrame with merged calls and puts data
    """
    
    options = tk.option_chain(exp_td_str)
    
    calls = options.calls
    puts = options.puts
    
    # Add optionType column
    calls["optionType"] = 'C'
    puts["optionType"] = 'P'
    
    # Merge calls and puts into a single dataframe
    exp_data = pd.concat(objs=[calls, puts], ignore_index=True)
    
    return exp_data
  
  tk = yf.Ticker(symbol)
  expirations = tk.options
  data = pd.DataFrame()

  for exp_td_str in expirations:
    exp_data = process_expiration(exp_td_str)
    exp_data["expirationDate"] = pd.to_datetime(exp_td_str)
    exp_data["daysToExpiration"] = (exp_data["expirationDate"] - pd.Timestamp("today")).dt.days
    data = pd.concat(objs=[data, exp_data], ignore_index=True)

  data["underlyingSymbol"] = symbol
  data["stockPrice"] = np.round(tk.history()['Close'].iloc[-1], 2)
  data["riskFreeRate"] = yf.Ticker("^IRX").history()['Close'].iloc[-1] / 100

  return data

In [39]:
def black_scholes(df, S_col, K_col, T_col, r_col, sigma_col, type_col):
  """
  df          - DataFrame containing the data
  S_col       - name of the column with the underlying asset price
  K_col       - strike price
  T_col       - time to expiration (in days)
  r_col       - risk-free interest rate
  sigma_col   - implied volatility
  type_col    - 'C' for call or 'P' for put
  """

  T = df[T_col] / 360  # convert days to years
  S = df[S_col]
  K = df[K_col]
  r = df[r_col]
  sigma = df[sigma_col]

  d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
  d2 = d1 - sigma * np.sqrt(T)

  call = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
  put = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

  df["bsPrice"] = np.round(np.where(df[type_col].str.upper() == "C", call, put), 2)
  return df


In [40]:
df = get_data(symbol)
df = black_scholes(df, "stockPrice", "strike", "daysToExpiration", "riskFreeRate", "impliedVolatility", "optionType")

In [61]:
df

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,inTheMoney,contractSize,currency,optionType,expirationDate,daysToExpiration,underlyingSymbol,stockPrice,riskFreeRate,bsPrice
0,AAPL260116C00005000,2026-01-08 20:25:15+00:00,5.0,253.12,252.55,256.15,0.000000,0.000000,16.0,60.0,...,True,REGULAR,USD,C,2026-01-16,4,AAPL,259.37,0.03513,255.22
1,AAPL260116C00010000,2026-01-08 20:25:15+00:00,10.0,247.98,247.55,251.30,0.000000,0.000000,2.0,29.0,...,True,REGULAR,USD,C,2026-01-16,4,AAPL,259.37,0.03513,249.38
2,AAPL260116C00015000,2026-01-09 16:04:55+00:00,15.0,243.06,242.55,246.30,0.080002,0.032925,2.0,2.0,...,True,REGULAR,USD,C,2026-01-16,4,AAPL,259.37,0.03513,244.38
3,AAPL260116C00020000,2026-01-09 16:04:55+00:00,20.0,238.10,238.25,240.95,0.250000,0.105108,2.0,138.0,...,True,REGULAR,USD,C,2026-01-16,4,AAPL,259.37,0.03513,239.42
4,AAPL260116C00025000,2025-12-22 18:27:04+00:00,25.0,245.99,232.60,236.35,0.000000,0.000000,2.0,15.0,...,True,REGULAR,USD,C,2026-01-16,4,AAPL,259.37,0.03513,234.39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2209,AAPL280317P00340000,2026-01-05 19:42:26+00:00,340.0,78.92,83.55,85.50,0.000000,0.000000,6.0,3.0,...,True,REGULAR,USD,P,2028-03-17,795,AAPL,259.37,0.03513,63.94
2210,AAPL280317P00350000,2026-01-08 20:43:04+00:00,350.0,93.75,91.90,94.05,0.000000,0.000000,195.0,196.0,...,True,REGULAR,USD,P,2028-03-17,795,AAPL,259.37,0.03513,71.02
2211,AAPL280317P00380000,2026-01-06 15:45:59+00:00,380.0,115.00,118.00,123.00,0.000000,0.000000,1.0,1.0,...,True,REGULAR,USD,P,2028-03-17,795,AAPL,259.37,0.03513,96.86
2212,AAPL280317P00400000,2025-12-29 19:13:20+00:00,400.0,126.65,138.50,143.00,0.000000,0.000000,2.0,0.0,...,True,REGULAR,USD,P,2028-03-17,795,AAPL,259.37,0.03513,115.18
